# SWE-bench Solver-Demand Classification

Run the pipeline from a terminal with `python run.py`. This notebook is a compact viewer for the joined evidence, draft taxonomy, final classifications, and review queue. It never makes API calls.

In [ ]:
from pathlib import Path
import importlib.util, shutil, subprocess, sys

HERE = Path.cwd()
if not (HERE / 'pipeline.py').exists():
    HERE = HERE / 'notebooks' / 'swebench_classification'
missing = [m for m in ['pandas', 'yaml'] if importlib.util.find_spec(m) is None]
if missing:
    uv = shutil.which('uv')
    if not uv:
        raise RuntimeError('uv is required')
    subprocess.check_call([uv, 'pip', 'install', '--python', sys.executable, '-r', str(HERE / 'requirements.txt')])
sys.path.insert(0, str(HERE))

import pandas as pd
from IPython.display import Markdown, display
from pipeline import load_config

config = load_config(HERE / 'config.yaml')
OUTPUT = Path(config['output_dir'])
OUTPUT

## Joined issue, problem statement, PR, and SWE-bench evidence

In [ ]:
enriched_path = OUTPUT / 'enriched_cases.csv'
enriched = pd.read_csv(enriched_path, dtype=str, keep_default_na=False)
print(f'{len(enriched)} cases; {enriched.case_id.nunique()} unique IDs')
enriched[['case_id', 'repo', 'problem_statement']].head(10)

## Human taxonomy review

In [ ]:
review_path = OUTPUT / 'taxonomy_review.md'
display(Markdown(review_path.read_text() if review_path.exists() else 'Run `python run.py --phase prepare` first.'))

## Final solver-demand distribution

In [ ]:
classifications_path = OUTPUT / 'case_classifications.csv'
if classifications_path.exists():
    classifications = pd.read_csv(classifications_path, dtype=str, keep_default_na=False)
    display(classifications['primary_solver_demand_class'].value_counts().rename_axis('class').to_frame('cases'))
    display(classifications[['interpretation_demand', 'diagnosis_demand', 'implementation_demand', 'verification_demand']].apply(pd.Series.value_counts).fillna(0).astype(int))
else:
    display(Markdown('Freeze the reviewed taxonomy and run the final phase first.'))

## Cases needing human review

In [ ]:
queue_path = OUTPUT / 'review_queue.csv'
pd.read_csv(queue_path, dtype=str, keep_default_na=False) if queue_path.exists() else 'No final review queue yet.'